### Funciona para Nubank para as tabelas de crédito pessoal e de cartão. 

> Ainda é interessante deixar isso automático, até na definição de página

O que falta?
* corrija algoritmo para reconhecer - - - 
* coloque também para reconhecer CP
* melhorar extração de trimestres ( deixar melhor )

Desafios:
* tirar a necessidade de colocar a página específica
* capacidade de extrair qualquer planilha contida no PDF

In [13]:
# Nossas opções:
# Nubank, Inter, Nubank, BB, Santander, Itaú ( todos tem textos selecionáveis ). Resultado é definição de uma janela

# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

## 14. Extração de dados para Empréstimo a Clientes 

In [22]:
# Refere-se a seção 14

with pdfplumber.open("Demonstrações Financeiras 3T25.pdf") as pdf:
    page = pdf.pages[19]  # página 20 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios")
end = text.find("Total", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]


c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios
30/09/2025 31/12/2024
Provisão Índice de Provisão Índice de
Exposição para perdas cobertura Exposição para perdas cobertura
bruta % de crédito % (%) bruta % de crédito % (%)
Forte (PD < 5%) 3.324.847 32,1% 41.531 3,0% 1,2% 1.954.790 31,9% 19.761 2,4% 1,0%
Estágio 1 3.277.840 98,6% 41.348 99,6% 1,3% 1.883.302 96,3% 18.678 94,5% 1,0%
Estágio 2 47.007 1,4% 183 0,4% 0,4% 71.488 3,7% 1.083 5,5% 1,5%
Satisfatório
3.486.996 33,7% 188.427 13,7% 5,4% 2.101.425 34,4% 113.253 14,3% 5,4%
(5% ≤ PD ≤ 20%)
Estágio 1 3.414.856 97,9% 185.173 98,3% 5,4% 1.855.922 88,3% 97.439 86,0% 5,3%
Estágio 2 72.140 2,1% 3.254 1,7% 4,5% 245.503 11,7% 15.814 14,0% 6,4%
Risco maior
3.550.129 34,2% 1.150.256 83,3% 32,4% 2.060.240 33,7% 661.556 83,3% 32,1%
(PD > 20%)
Estágio 1 1.511.520 42,6% 188.999 16,4% 12,5% 989.134 48,0% 123.189 18,6% 12,5%
Estágio 2 1.402.393 39,5% 537.058 46,7% 38,3% 737.425 35,8% 308.123 46,6% 41,8%
Estágio 3 636.216 17,9

### Retirando elementos desnecessários da extração

In [23]:
def limpar_linhas_estagio(linhas):

    resultado = []
    ignorar_ate_estagio = False

    PD_HEADERS = ("Forte", "Satisfatório", "Risco maior") # Define título

    for linha in linhas:

        if linha.startswith(PD_HEADERS): # ignora tudo, menos estágios
            ignorar_ate_estagio = True
            continue

        if ignorar_ate_estagio:
            if linha.startswith("Estágio"): # Define sessão que não deve ser excluída
                ignorar_ate_estagio = False
            else:
                continue

        if not linha.startswith("Estágio"):
            continue

        tokens = linha.split()

        # remove porcentagens
        tokens_sem_percent = [t for t in tokens if "%" not in t]

        # ignora "Estágio" e o número do estágio
        tokens_dados = tokens_sem_percent[2:]

        numeros = [
            t for t in tokens_dados
            if re.fullmatch(r"\d{1,3}(?:\.\d{3})*|–", t)
        ]
        # --------------------------------

        estagio = " ".join(tokens_sem_percent[:2])  # Estágio 1 / 2 / 3

        resultado.append(
            " ".join([estagio] + numeros)
        )

    return resultado



In [24]:
# há apenas um erro: você deve ensinar o código a lidar com -. Está dando como nulo 4T25 para estágio 2 justamente pelo fato que está pegando
# o que está embaixo


In [25]:
linhas_limpa = limpar_linhas_estagio(linhas)

for l in linhas_limpa:
    print(l)

Estágio 1 3.277.840 41.348 1.883.302 18.678
Estágio 2 47.007 183 71.488 1.083
Estágio 1 3.414.856 185.173 1.855.922 97.439
Estágio 2 72.140 3.254 245.503 15.814
Estágio 1 1.511.520 188.999 989.134 123.189
Estágio 2 1.402.393 537.058 737.425 308.123
Estágio 3 636.216 424.199 333.681 230.244


### Extração do trimestre

In [26]:
from datetime import datetime

def datas_para_trimestres(texto):
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    trimestres = []

    for d in datas:
        dt = datetime.strptime(d, "%d/%m/%Y")
        trimestre = (dt.month - 1) // 3 + 1
        ano = str(dt.year)[-2:]
        trimestres.append(f"{trimestre}T{ano}")

    return trimestres

### Padrão fixo para colunas do dataframe

In [27]:
PD_PADRAO = [
    "Forte (PD < 5%)",
    "Forte (PD < 5%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Satisfatório (5% ≤ PD ≤ 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
    "Risco maior (PD > 20%)",
]

ESTAGIO_PADRAO = [1, 2, 1, 2, 1, 2, 3]

### Para as colunas de Exposição Bruta e PE

In [28]:
def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]

    def conv(x):
        if x == "–":
            return None
        return float(x.replace(".", "").replace(",", "."))

    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }


In [31]:

def construir_dataframe(
    linhas_estagio,
    texto_pdf,
    banco="Nubank",
    produto="CP"
):

    trimestre_atual, trimestre_anterior = datas_para_trimestres(texto_pdf)

    registros = []

    for i, linha in enumerate(linhas_estagio):

        dados = parse_linha_estagio(linha)

        # Linha do trimestre atual
        registros.append({
            "ano": trimestre_atual,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_atual"],
            "PE": dados["pe_atual"]
        })

        # Linha do trimestre anterior
        registros.append({
            "ano": trimestre_anterior,
            "banco": banco.lower(),
            "produto": produto,
            "PD": PD_PADRAO[i],
            "Estágio": ESTAGIO_PADRAO[i],
            "Exposição Bruta": dados["exp_bruta_anterior"],
            "PE": dados["pe_anterior"]
        })

    df = pd.DataFrame(registros)

    return df


In [32]:
df = construir_dataframe(
    linhas_estagio=linhas_limpa,
    texto_pdf=trecho,
    banco="Nubank",
    produto="CP"
)

df

,ano,banco,produto,PD,Estágio,Exposição Bruta,PE
0,3T25,nubank,CP,Forte (PD < 5%),1,3277840.0,41348.0
1,4T24,nubank,CP,Forte (PD < 5%),1,1883302.0,18678.0
2,3T25,nubank,CP,Forte (PD < 5%),2,47007.0,183.0
3,4T24,nubank,CP,Forte (PD < 5%),2,71488.0,1083.0
4,3T25,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),1,3414856.0,185173.0
5,4T24,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),1,1855922.0,97439.0
6,3T25,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),2,72140.0,3254.0
7,4T24,nubank,CP,Satisfatório (5% ≤ PD ≤ 20%),2,245503.0,15814.0
8,3T25,nubank,CP,Risco maior (PD > 20%),1,1511520.0,188999.0
9,4T24,nubank,CP,Risco maior (PD > 20%),1,989134.0,123189.0
